# Method — brapi-py

Interactive notebook for exploring the BrAPI **Method** endpoint.
Run each cell with **Shift+Enter** (or the ▶ button).

> **First time?** Select a Python kernel with the **Select Kernel** button (top-right).  
> If `brapi-py` is not installed, uncomment the `pip install` line in the next cell.

In [ ]:
# Uncomment to install / upgrade brapi-py
# %pip install brapi-py
import brapi
print('brapi-py version:', brapi.__version__)

## 1 — Connection
Edit the cell below with your server credentials, then run it.
The `with` block is optional — the client stays open for the whole notebook.

In [ ]:
from brapi import BrapiClient

# ── Edit these ────────────────────────────────────────────────────────────
BASE_URL       = 'https://brapi.example.com'
TOKEN_ENDPOINT = 'https://auth.example.com/token'
CLIENT_ID      = 'my-client'
CLIENT_SECRET  = 'my-secret'
# ─────────────────────────────────────────────────────────────────────────

client = BrapiClient(
    base_url=BASE_URL,
    token_endpoint=TOKEN_ENDPOINT,
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
)
print('Client ready →', client)

## 3 — List  (`GET /methods`)

Simpler GET endpoint — same filter state, mapped to query-string params.
Useful when the server does not support the search endpoint, or for quick lookups.

In [ ]:
# GET /methods — list records
df = (
    client.method
    .common_crop_names(['Tomatillo'])
    .list()
    .to_df()
)
print(f'{len(df)} records')
df.head()

## 4 — Single-record CRUD

Get, create, update, and delete individual records.

In [ ]:
# GET /methods/ {id}  — fetch one record by primary key
METHOD_DBID = 'example-001'   # ← change to a real ID

record = client.method.get_by_id(METHOD_DBID)
print(record)

In [ ]:
from brapi.entities.generated_method import Method

# POST /methods — create a new record
new_method = Method(
    methodDbId='',  # assigned by server
    methodName='Example Method',
)
created = client.method.create(new_method)
print('Created methodDbId:', created.methodDbId)

In [ ]:
# PUT /methods/ {id} — update the record created above
# (run the create cell first to populate 'created')
created.bibliographicalReference = 'updated value'
updated = client.method.update(created.methodDbId, created)
print('Updated:', updated.methodDbId)

## 6 — DataFrame exploration

Useful pandas operations once you have a DataFrame.

In [ ]:
# Reload a fresh DataFrame
df = (
    client.method
    .common_crop_names(['Tomatillo'])
    .list()
    .to_df()
)

print('Columns:', df.columns.tolist())
print('Shape:', df.shape)
df.head()

In [ ]:
# Value distribution for 'bibliographicalReference'
if 'bibliographicalReference' in df.columns:
    df['bibliographicalReference'].value_counts().head(10)
else:
    print('Column not present in this result set')

In [ ]:
import json, pandas as pd

# Explode the 'externalReferences' JSON column into separate rows
col = 'externalReferences'
if col in df.columns:
    id_col = 'methodDbId'
    exploded = (
        df.filter(items=[id_col, col])
        .dropna(subset=[col])
        .assign(**{col: lambda d: d[col].apply(json.loads)})
        .explode(col)
    )
    exploded.head(10)
else:
    print(f'Column {col!r} not present in this result set')

## 7 — Error handling

In [ ]:
import httpx

# 404 — record not found
try:
    client.method.get_by_id('does-not-exist-xyz')
except httpx.HTTPStatusError as e:
    print(f'HTTP {e.response.status_code}: {e.request.url}')


# ValueError from create_many with only one item
try:
    client.method.create_many([])
except (ValueError, AttributeError) as e:
    print('Error:', e)

## 8 — Clean up
Close the HTTP client when done to release the connection pool.

In [ ]:
client.close()
print('Connection closed')